# M2: Entrenamiento de Segmentación de Instancias de Vehículos (YOLO11-seg)
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio Final de Hito: Sistema Inteligente de Control de Peaje

Este notebook implementa el pipeline completo para el modelo **M2 (Segmentación de Instancias)** que clasifica y segmenta las **5 categorías tarifarias del peaje**:
- `moto` (Motocicleta)
- `auto` (Auto particular / Vagoneta / Camioneta)
- `minibus` (Minibús / Van)
- `camion` (Camión de carga)
- `bus` (Bus / Flota)

### Flujo de trabajo en este Notebook:
1. **Pre-etiquetado inteligente con YOLOE (Vocabulario Abierto):** Permite tomar imágenes propias (como las de `Placas_en_mi_caso` y capturas de vehículos) y generar polígonos `YOLO-seg` de las 5 clases para subirlos a **Roboflow**.
2. **Descarga del Dataset desde Roboflow:** Conecta mediante `ROBOFLOW_API_KEY` (leída desde *Colab Secrets* o variable de entorno, sin exponer credenciales en código) y descarga el dataset en formato `yolov11`.
3. **Entrenamiento con Ultralytics `yolo11s-seg.pt`:** Entrena el modelo de segmentación de instancias en GPU T4.
4. **Evaluación y Métricas para el Informe Técnico:** Calcula y grafica `mAP50`, `mAP50-95` (Box y Mask), Precisión, Recall, **Matriz de Confusión** y **Área de Máscara (`area_mascara`)** por instancia.
5. **Exportación de Pesos (`m2_seg_best.pt`)** a Google Drive para que el servidor unificado de inferencia (`/inferencia`) los cargue directamente.

## 1. Instalación de dependencias y verificación de GPU

In [ ]:
!nvidia-smi -L || echo "ATENCIÓN: Active la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno"
!pip -q install -U ultralytics roboflow pandas matplotlib seaborn
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'

## 2. Configuración general y clases del peaje

In [ ]:
import os, glob, time, shutil, json, cv2, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo activo:', DEVICE)

# las 5 categorías obligatorias según la guía del laboratorio (Sección 2 y 4.7)
CLASES_M2 = ['moto', 'auto', 'minibus', 'camion', 'bus']
ID_CLASE = {c: i for i, c in enumerate(CLASES_M2)}

# Prompts de vocabulario abierto para pre-etiquetado con YOLOE-seg
PROMPTS_YOLOE = {
    'motorcycle': 'moto', 'motorbike': 'moto', 'scooter': 'moto',
    'car': 'auto', 'sedan': 'auto', 'suv': 'auto', 'pickup truck': 'auto',
    'minibus': 'minibus', 'van': 'minibus', 'minivan': 'minibus',
    'truck': 'camion', 'cargo truck': 'camion', 'semi truck': 'camion',
    'bus': 'bus', 'coach bus': 'bus'
}

# Normalizador de nombres de clases por si el dataset de Roboflow tiene variantes en inglés o español
NORMALIZAR_CLASE = {
    'moto': 'moto', 'motocicleta': 'moto', 'motorcycle': 'moto', 'motorbike': 'moto',
    'auto': 'auto', 'car': 'auto', 'automovil': 'auto', 'auto particular': 'auto', 'suv': 'auto', 'vagoneta': 'auto',
    'minibus': 'minibus', 'minibús': 'minibus', 'van': 'minibus', 'trufi': 'minibus',
    'camion': 'camion', 'camión': 'camion', 'truck': 'camion',
    'bus': 'bus', 'flota': 'bus'
}

COLORES = {
    'moto': (155, 89, 182),
    'auto': (52, 152, 219),
    'minibus': (46, 204, 113),
    'camion': (243, 156, 18),
    'bus': (231, 76, 60)
}

RUTA_PESOS_M2 = '/content/runs/segment/m2_vehiculos/weights/best.pt'

## 3. (Opcional) Pre-etiquetado automático con YOLOE-seg para ampliar dataset propio en Roboflow
Suba un archivo `.zip` o imágenes sueltas de vehículos para generar automáticamente las máscaras poligonales de las 5 categorías (`moto`, `auto`, `minibus`, `camion`, `bus`), listas para importar y refinar en Roboflow.

In [ ]:
PREETIQUETAR_PROPIAS = False  # Cambiar a True si desea generar etiquetas iniciales para subir a Roboflow

if PREETIQUETAR_PROPIAS:
    from google.colab import files
    os.makedirs('/content/raw_imgs', exist_ok=True)
    for nombre in files.upload().keys():
        if nombre.lower().endswith('.zip'):
            !unzip -qo "$nombre" -d /content/raw_imgs
        else:
            shutil.move(nombre, '/content/raw_imgs/' + nombre)

    nombres_prompts = list(PROMPTS_YOLOE.keys())
    yoloe = YOLOE('yoloe-11l-seg.pt')
    yoloe.set_classes(nombres_prompts, yoloe.get_text_pe(nombres_prompts))

    os.makedirs('/content/preetiquetas_m2/images', exist_ok=True)
    os.makedirs('/content/preetiquetas_m2/labels', exist_ok=True)

    rutas_imgs = sorted(p for p in glob.glob('/content/raw_imgs/**/*', recursive=True)
                        if p.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')))
    total_instancias = 0
    for idx, ruta in enumerate(rutas_imgs):
        im = cv2.imread(ruta)
        if im is None:
            continue
        H, W = im.shape[:2]
        r = yoloe.predict(im, conf=0.25, device=DEVICE, agnostic_nms=True, verbose=False)[0]
        lineas = []
        if r.masks is not None:
            for poly, c in zip(r.masks.xy, r.boxes.cls.cpu().numpy().astype(int)):
                clase_dest = PROMPTS_YOLOE.get(r.names[c])
                if clase_dest and len(poly) >= 3:
                    coords = ' '.join(f'{min(max(x / W, 0), 1):.5f} {min(max(y / H, 0), 1):.5f}' for x, y in poly)
                    lineas.append(f'{ID_CLASE[clase_dest]} {coords}')
                    total_instancias += 1
        base = f'{idx:04d}_' + os.path.splitext(os.path.basename(ruta))[0]
        cv2.imwrite(f'/content/preetiquetas_m2/images/{base}.jpg', im)
        with open(f'/content/preetiquetas_m2/labels/{base}.txt', 'w') as f:
            f.write('\n'.join(lineas))

    with open('/content/preetiquetas_m2/data.yaml', 'w') as f:
        f.write(f'names: {CLASES_M2}\nnc: {len(CLASES_M2)}\n')

    !cd /content && zip -qr preetiquetas_m2_roboflow.zip preetiquetas_m2
    print(f'Pre-etiquetadas {len(rutas_imgs)} imágenes con {total_instancias} máscaras.')
    files.download('/content/preetiquetas_m2_roboflow.zip')

## 4. Descarga del Dataset desde Roboflow (o generación de dataset de respaldo verificable)
Configura en el panel izquierdo de Colab (**Secrets / Llave 🔑**) los secretos:
- `ROBOFLOW_API_KEY`
- `ROBOFLOW_WORKSPACE`
- `ROBOFLOW_PROJECT_M2`
- `ROBOFLOW_VERSION_M2` (por defecto `1`)

In [ ]:
from google.colab import userdata

def obtener_secreto(nombre, por_defecto=''):
    try:
        val = userdata.get(nombre)
        return val if val else os.environ.get(nombre, por_defecto)
    except Exception:
        return os.environ.get(nombre, por_defecto)

RF_API_KEY = obtener_secreto('ROBOFLOW_API_KEY')
RF_WORKSPACE = obtener_secreto('ROBOFLOW_WORKSPACE', 'peaje-iam811')
RF_PROJECT = obtener_secreto('ROBOFLOW_PROJECT_M2', 'vehiculos-peaje-seg')
RF_VERSION = int(obtener_secreto('ROBOFLOW_VERSION_M2', '1'))

DATA_YAML_M2 = None

if RF_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=RF_API_KEY)
    dataset = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download('yolov11')
    DATA_YAML_M2 = os.path.join(dataset.location, 'data.yaml')
    print('Dataset M2 descargado desde Roboflow en:', DATA_YAML_M2)
else:
    print('AVISO: No se detectó ROBOFLOW_API_KEY en Colab Secrets. Configure ROBOFLOW_API_KEY para descargar su dataset de Roboflow.')

## 5. Entrenamiento de M2 (`yolo11s-seg.pt`)

In [ ]:
assert DATA_YAML_M2 and os.path.exists(DATA_YAML_M2), 'Debe definir DATA_YAML_M2 descargando el dataset de Roboflow o subiendo su dataset.'

modelo_m2 = YOLO('yolo11s-seg.pt')
resultados_train = modelo_m2.train(
    data=DATA_YAML_M2,
    epochs=60,
    imgsz=640,
    batch=16,
    device=0 if DEVICE == 'cuda' else 'cpu',
    project='/content/runs/segment',
    name='m2_vehiculos',
    exist_ok=True,
    plots=True
)
print('Entrenamiento finalizado. Pesos guardados en:', RUTA_PESOS_M2)

## 6. Evaluación de Métricas Oficiales para el Informe Técnico (mAP, Matriz de Confusión y `area_mascara`)

In [ ]:
mejor_m2 = YOLO(RUTA_PESOS_M2)
metricas = mejor_m2.val(data=DATA_YAML_M2, split='val', plots=True)

resumen_metricas = pd.DataFrame([{
    'Modelo': 'M2 - YOLO11s-seg',
    'Box mAP@50': round(float(metricas.box.map50), 4),
    'Box mAP@50-95': round(float(metricas.box.map), 4),
    'Mask mAP@50': round(float(metricas.seg.map50), 4),
    'Mask mAP@50-95': round(float(metricas.seg.map), 4),
    'Precisión (Mask)': round(float(metricas.seg.mp), 4),
    'Recall (Mask)': round(float(metricas.seg.mr), 4)
}])
display(resumen_metricas)

# Mostrar matriz de confusión y curvas generadas por Ultralytics
for grafico in ['confusion_matrix_normalized.png', 'MaskPR_curve.png', 'results.png']:
    ruta_g = os.path.join('/content/runs/segment/m2_vehiculos', grafico)
    if os.path.exists(ruta_g):
        img_g = cv2.imread(ruta_g)
        plt.figure(figsize=(10, 7))
        plt.imshow(cv2.cvtColor(img_g, cv2.COLOR_BGR2RGB))
        plt.title(grafico)
        plt.axis('off')
        plt.show()

## 7. Prueba de salida en formato Endpoint del Peaje (`vehiculo: {categoria, confianza, area_mascara}`)
Verifica que la salida de M2 cumple exactamente con el diagrama de la página 5 de la guía del laboratorio.

In [ ]:
def inferir_m2_peaje(img_bgr, conf=0.30):
    m = YOLO(RUTA_PESOS_M2) if os.path.exists(RUTA_PESOS_M2) else YOLO('yolo11s-seg.pt')
    r = m.predict(img_bgr, conf=conf, device=DEVICE, agnostic_nms=True, verbose=False)[0]
    H, W = r.orig_shape
    area_img = float(H * W)
    instancias = []
    if r.masks is None or len(r.boxes) == 0:
        return {'vehiculo': {'categoria': 'auto', 'confianza': 0.0, 'area_mascara': 0.0}, 'instancias': []}

    for poly, c, s, b in zip(r.masks.xy, r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy()):
        nombre_raw = r.names[c].strip().lower()
        cat = NORMALIZAR_CLASE.get(nombre_raw)
        if not cat or len(poly) < 3:
            continue
        mask = np.zeros((H, W), np.uint8)
        cv2.fillPoly(mask, [poly.astype(np.int32)], 1)
        area_rel = round(float(mask.sum()) / area_img, 4)
        instancias.append({
            'categoria': cat,
            'confianza': round(float(s), 3),
            'area_mascara': area_rel,
            'bbox': [int(v) for v in b]
        })

    if not instancias:
        return {'vehiculo': {'categoria': 'auto', 'confianza': 0.0, 'area_mascara': 0.0}, 'instancias': []}

    # Selecciona el vehículo principal (mayor área de máscara en la caseta)
    principal = max(instancias, key=lambda x: x['area_mascara'])
    return {'vehiculo': principal, 'instancias': instancias}

!wget -q -O /content/test_bus.jpg https://ultralytics.com/images/bus.jpg
img_test = cv2.imread('/content/test_bus.jpg')
print(json.dumps(inferir_m2_peaje(img_test), indent=2, ensure_ascii=False))

## 8. Guardar pesos entrenados en Google Drive para el Servidor de Inferencia

In [ ]:
GUARDAR_EN_DRIVE = False  # Cambiar a True para copiar m2_seg_best.pt a Google Drive
if GUARDAR_EN_DRIVE and os.path.exists(RUTA_PESOS_M2):
    from google.colab import drive
    drive.mount('/content/drive')
    dir_destino = '/content/drive/MyDrive/IAM811_Peaje_Pesos'
    os.makedirs(dir_destino, exist_ok=True)
    shutil.copy(RUTA_PESOS_M2, os.path.join(dir_destino, 'm2_seg_best.pt'))
    print('Pesos M2 guardados en:', os.path.join(dir_destino, 'm2_seg_best.pt'))